In [149]:
import pandas as pd
df = pd.read_csv("./data/emails.csv")
print(len(df))
df.head()

517401


,file,message
0,allen-p/_sent_mail/1.,Message-ID: <18782981.1075855378110.JavaMail.e...
1,allen-p/_sent_mail/10.,Message-ID: <15464986.1075855378456.JavaMail.e...
2,allen-p/_sent_mail/100.,Message-ID: <24216240.1075855687451.JavaMail.e...
3,allen-p/_sent_mail/1000.,Message-ID: <13505866.1075863688222.JavaMail.e...
4,allen-p/_sent_mail/1001.,Message-ID: <30922949.1075863688243.JavaMail.e...


In [ ]:
print(df["message"][2])

In [151]:
import re

def parse_email(raw):

    fields = {}

    patterns = {
        "message_id": r"Message-ID:\s*(.*)",
        "date": r"Date:\s*(.*)",
        "from": r"From:\s*(.*)",
        "to": r"To:\s*(.*)",
        "subject": r"Subject:\s*(.*)"
    }

    for key, pattern in patterns.items():
        match = re.search(pattern, raw)
        fields[key] = match.group(1).strip() if match else ""

    return fields

In [152]:
parsed = df["message"].apply(parse_email)
meta_df = pd.json_normalize(parsed)

meta_df.head()

,message_id,date,from,to,subject
0,<18782981.1075855378110.JavaMail.evans@thyme>,"Mon, 14 May 2001 16:39:00 -0700 (PDT)",phillip.allen@enron.com,tim.belden@enron.com,Mime-Version: 1.0
1,<15464986.1075855378456.JavaMail.evans@thyme>,"Fri, 4 May 2001 13:51:00 -0700 (PDT)",phillip.allen@enron.com,john.lavorato@enron.com,Re:
2,<24216240.1075855687451.JavaMail.evans@thyme>,"Wed, 18 Oct 2000 03:00:00 -0700 (PDT)",phillip.allen@enron.com,leah.arsdall@enron.com,Re: test
3,<13505866.1075863688222.JavaMail.evans@thyme>,"Mon, 23 Oct 2000 06:13:00 -0700 (PDT)",phillip.allen@enron.com,randall.gay@enron.com,Mime-Version: 1.0
4,<30922949.1075863688243.JavaMail.evans@thyme>,"Thu, 31 Aug 2000 05:07:00 -0700 (PDT)",phillip.allen@enron.com,greg.piper@enron.com,Re: Hello


In [153]:
import re

def extract_body(raw):
    
    # split after the last header line
    body_start = re.split(r"\nX-FileName:.*\n", raw)

    if len(body_start) > 1:
        return body_start[1].strip()

    # fallback method
    parts = raw.split("\n\n", 1)
    if len(parts) > 1:
        return parts[1].strip()

    return ""

In [154]:
df["body"] = df["message"].apply(extract_body)

In [ ]:
df[["subject", "body"]].head()

In [156]:
print(df.columns)

Index(['file', 'message', 'body'], dtype='str')


In [157]:
print(df["message"].iloc[3])

Message-ID: <13505866.1075863688222.JavaMail.evans@thyme>
Date: Mon, 23 Oct 2000 06:13:00 -0700 (PDT)
From: phillip.allen@enron.com
To: randall.gay@enron.com
Subject: 
Mime-Version: 1.0
Content-Type: text/plain; charset=us-ascii
Content-Transfer-Encoding: 7bit
X-From: Phillip K Allen
X-To: Randall L Gay
X-cc: 
X-bcc: 
X-Folder: \Phillip_Allen_Dec2000\Notes Folders\'sent mail
X-Origin: Allen-P
X-FileName: pallen.nsf

Randy,

 Can you send me a schedule of the salary and level of everyone in the 
scheduling group.  Plus your thoughts on any changes that need to be made.  
(Patti S for example)

Phillip


In [158]:
import re

def parse_email(raw):

    data = {}

    fields = [
        "Message-ID",
        "Date",
        "From",
        "To",
        "Subject",
        "X-From",
        "X-To",
        "X-Folder"
    ]

    for field in fields:
        match = re.search(rf"{field}:\s*(.*)", raw)
        data[field.lower().replace("-", "_")] = match.group(1).strip() if match else ""

    # Extract body
    parts = raw.split("\n\n", 1)
    body = parts[1] if len(parts) > 1 else ""

    data["body"] = body.strip()

    return data

In [159]:
parsed = df["message"].apply(parse_email)
email_df = pd.json_normalize(parsed)

email_df.head()

,message_id,date,from,to,subject,x_from,x_to,x_folder,body
0,<18782981.1075855378110.JavaMail.evans@thyme>,"Mon, 14 May 2001 16:39:00 -0700 (PDT)",phillip.allen@enron.com,tim.belden@enron.com,Mime-Version: 1.0,Phillip K Allen,Tim Belden <Tim Belden/Enron@EnronXGate>,"\Phillip_Allen_Jan2002_1\Allen, Phillip K.\'Se...",Here is our forecast
1,<15464986.1075855378456.JavaMail.evans@thyme>,"Fri, 4 May 2001 13:51:00 -0700 (PDT)",phillip.allen@enron.com,john.lavorato@enron.com,Re:,Phillip K Allen,John J Lavorato <John J Lavorato/ENRON@enronXg...,"\Phillip_Allen_Jan2002_1\Allen, Phillip K.\'Se...",Traveling to have a business meeting takes the...
2,<24216240.1075855687451.JavaMail.evans@thyme>,"Wed, 18 Oct 2000 03:00:00 -0700 (PDT)",phillip.allen@enron.com,leah.arsdall@enron.com,Re: test,Phillip K Allen,Leah Van Arsdall,\Phillip_Allen_Dec2000\Notes Folders\'sent mail,test successful. way to go!!!
3,<13505866.1075863688222.JavaMail.evans@thyme>,"Mon, 23 Oct 2000 06:13:00 -0700 (PDT)",phillip.allen@enron.com,randall.gay@enron.com,Mime-Version: 1.0,Phillip K Allen,Randall L Gay,\Phillip_Allen_Dec2000\Notes Folders\'sent mail,"Randy,\n\n Can you send me a schedule of the s..."
4,<30922949.1075863688243.JavaMail.evans@thyme>,"Thu, 31 Aug 2000 05:07:00 -0700 (PDT)",phillip.allen@enron.com,greg.piper@enron.com,Re: Hello,Phillip K Allen,Greg Piper,\Phillip_Allen_Dec2000\Notes Folders\'sent mail,Let's shoot for Tuesday at 11:45.


In [160]:
df_clean = pd.concat([df, email_df], axis=1)

In [161]:
df_clean[["from", "to", "subject", "body"]].head()

,from,to,subject,body,body
0,phillip.allen@enron.com,tim.belden@enron.com,Mime-Version: 1.0,Here is our forecast,Here is our forecast
1,phillip.allen@enron.com,john.lavorato@enron.com,Re:,Traveling to have a business meeting takes the...,Traveling to have a business meeting takes the...
2,phillip.allen@enron.com,leah.arsdall@enron.com,Re: test,test successful. way to go!!!,test successful. way to go!!!
3,phillip.allen@enron.com,randall.gay@enron.com,Mime-Version: 1.0,"Randy,\n\n Can you send me a schedule of the s...","Randy,\n\n Can you send me a schedule of the s..."
4,phillip.allen@enron.com,greg.piper@enron.com,Re: Hello,Let's shoot for Tuesday at 11:45.,Let's shoot for Tuesday at 11:45.


In [162]:
df_clean = df_clean.loc[:, ~df_clean.columns.duplicated()]

In [163]:
df_clean.columns

Index(['file', 'message', 'body', 'message_id', 'date', 'from', 'to',
       'subject', 'x_from', 'x_to', 'x_folder'],
      dtype='str')

In [164]:
df_clean[["from", "to", "subject", "body"]].head()

,from,to,subject,body
0,phillip.allen@enron.com,tim.belden@enron.com,Mime-Version: 1.0,Here is our forecast
1,phillip.allen@enron.com,john.lavorato@enron.com,Re:,Traveling to have a business meeting takes the...
2,phillip.allen@enron.com,leah.arsdall@enron.com,Re: test,test successful. way to go!!!
3,phillip.allen@enron.com,randall.gay@enron.com,Mime-Version: 1.0,"Randy,\n\n Can you send me a schedule of the s..."
4,phillip.allen@enron.com,greg.piper@enron.com,Re: Hello,Let's shoot for Tuesday at 11:45.


In [165]:
def extract_subject(raw):

    for line in raw.split("\n"):
        if line.startswith("Subject:"):
            return line.replace("Subject:", "").strip()

    return ""

In [166]:
df["subject"] = df["message"].apply(extract_subject)

In [167]:
print(df_clean.columns)

Index(['file', 'message', 'body', 'message_id', 'date', 'from', 'to',
       'subject', 'x_from', 'x_to', 'x_folder'],
      dtype='str')


In [168]:
print(df_clean["body"].isna().sum())

0


In [169]:
print(df_clean["subject"].value_counts().head())

subject
Mime-Version: 1.0                                        17015
RE:                                                       6477
Re:                                                       6308
Demand Ken Lay Donate Proceeds from Enron Stock Sales     1124
FW:                                                        938
Name: count, dtype: int64


In [170]:
import re

def normalize_subject(subject):

    if not isinstance(subject, str):
        return ""

    subject = subject.lower().strip()

    # remove reply prefixes repeatedly
    subject = re.sub(r'^(re|fw|fwd)\s*:\s*', '', subject)

    # remove extra spaces
    subject = re.sub(r'\s+', ' ', subject)

    return subject.strip()

In [171]:
df_clean["subject_clean"] = df_clean["subject"].apply(normalize_subject)

In [172]:
df_clean[["subject", "subject_clean"]].head(10)

,subject,subject_clean
0,Mime-Version: 1.0,mime-version: 1.0
1,Re:,
2,Re: test,test
3,Mime-Version: 1.0,mime-version: 1.0
4,Re: Hello,hello
5,Re: Hello,hello
6,Mime-Version: 1.0,mime-version: 1.0
7,Re: PRC review - phone calls,prc review - phone calls
8,Re: High Speed Internet Access,high speed internet access
9,FW: fixed forward or other Collar floor gas pr...,fixed forward or other collar floor gas price ...


In [173]:
df_clean = df_clean[df_clean["subject_clean"] != ""]

In [174]:
df_clean = df_clean[~df_clean["subject_clean"].isin(["re", "test"])]

In [175]:
df_clean[["subject", "subject_clean"]].head(10)

,subject,subject_clean
0,Mime-Version: 1.0,mime-version: 1.0
3,Mime-Version: 1.0,mime-version: 1.0
4,Re: Hello,hello
5,Re: Hello,hello
6,Mime-Version: 1.0,mime-version: 1.0
7,Re: PRC review - phone calls,prc review - phone calls
8,Re: High Speed Internet Access,high speed internet access
9,FW: fixed forward or other Collar floor gas pr...,fixed forward or other collar floor gas price ...
10,Re: FW: fixed forward or other Collar floor ga...,fw: fixed forward or other collar floor gas pr...
11,Mime-Version: 1.0,mime-version: 1.0


In [176]:
threads = df_clean.groupby("subject_clean")
thread_sizes = threads.size().sort_values(ascending=False)

In [177]:
thread_sizes.head(20)

subject_clean
mime-version: 1.0                                        17015
demand ken lay donate proceeds from enron stock sales     1124
schedule crawler: hourahead failure                        900
enron mentions                                             835
schedule crawler: hourahead failure <codesite>             800
entouch newsletter                                         554
(no subject)                                               544
lunch                                                      531
hey                                                        458
meeting                                                    453
hi                                                         443
organizational announcement                                431
organizational changes                                     416
congratulations                                            415
apb checkout                                               387
energy issues                            

In [178]:
valid_threads = thread_sizes[thread_sizes >= 3]

In [179]:
selected_subjects = valid_threads.head(15).index

dataset = df_clean[df_clean["subject_clean"].isin(selected_subjects)]

In [180]:
thread_map = {
    subject: f"T-{i:03d}"
    for i, subject in enumerate(selected_subjects, start=1)
}

dataset["thread_id"] = dataset["subject_clean"].map(thread_map)

In [181]:
print("Threads:", dataset["thread_id"].nunique())
print("Messages:", len(dataset))

Threads: 15
Messages: 25306


In [ ]:
dataset["date"] = dataset["date"].astype(str)

dataset.to_json(
    "data/threaded_emails.json",
    orient="records",
    indent=2
)